# پیوند مخابراتی با دهانهٔ محدود

ابتدا پیش‌بینی کنید: آیا دو برابر شدن تفکیک شبکه، توان دریافتی را بیشتر می‌کند؟ آیا باریک‌تر شدن پرتو همیشه نرخ خالص را بهبود می‌دهد؟ این دفترچه هندسه، دامنهٔ کانال، نویز و آموزش را جدا می‌کند. مدل اسکالر معرفی‌شده در [فصل](../chapters/08-communications.md) اجرا می‌شود؛ فراسطح طرح‌دار حل نمی‌شود.

سلول‌ها را به‌ترتیب اجرا کنید. پایتون ۳٫۱۲، نامپای و مت‌پلات‌لیب برای دفترچه کافی‌اند؛ ارزیابی مستقل از سای‌پای نیز استفاده می‌کند. پیش از تغییر پارامترها دادهٔ مرجع را نگه دارید؛ سلول آخر data/chapter-08/reference.json را می‌نویسد. برچسب‌های علمی نمودارها در دو زبان مشترک‌اند.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import sys, hashlib, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/'simulations').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from simulations.communication_link import (Link, C0, aperture, paths, phase_command,
    channel, noise_power, metrics, error_expectation, training)
p = Link()  # All physical, noise and sampling parameters are editable here.
commands = {'focus': phase_command(p), 'plane': phase_command(p, mode='plane'),
            '2bit': phase_command(p, bits=2)}
xyz, distance, weights = paths(p)
h0 = channel(commands['focus'], p)
gain0 = abs(h0)**2
db = lambda value: 10*np.log10(np.maximum(value, 1e-30))
plt.rcParams.update({'figure.dpi':120, 'font.size':10, 'axes.spines.top':False,
                     'axes.spines.right':False, 'axes.grid':True, 'grid.alpha':.2})
colors = ['#087f8c','#e18431','#7252a1']
evidence = {'metadata': {'parameters':asdict(p), 'model':'declared scalar projected-aperture channel',
    'source_sha256':hashlib.sha256((ROOT/'simulations/communication_link.py').read_text(encoding='utf-8').encode()).hexdigest()}}
print(asdict(p))
print('Channel gain (dB):', db(gain0), 'Noise (dBm):', db(noise_power(p)*1000))
print('SNR (dB):', db(metrics(h0,p)['snr']), 'Gross rate (Mbps):', metrics(h0,p)['net_Mbps'])

## ۱. دهانهٔ برنامه‌ریزی‌شده و صفحهٔ گیرنده

پنل چپ فاز فرمان روی **دهانهٔ مربعی ۰٫۴۸ متری** در z = 0 را نشان می‌دهد؛ نقشهٔ ساخت سلول نیست. در پنل راست فرمان ثابت است و گیرنده در امتداد x و در z = 12 m جابه‌جا می‌شود. طرح کروی دقیقاً برای هدف تعیین‌شده ساخته شده است. قانون صفحه‌ای انحنای موج را حذف می‌کند و قانون دوبیتی فاز کروی را به چهار حالت گرد می‌کند. همهٔ منحنی‌ها را با مرجع مشترک هدف ایده‌آل مقایسه کنید.

In [ ]:
xs = np.linspace(4,12,241)
profiles = {key:np.array([abs(channel(g,p,(x,0,p.rx_m[2])))**2 for x in xs]) for key,g in commands.items()}
fig,ax=plt.subplots(1,2,figsize=(11.5,4.2),layout='constrained')
im=ax[0].imshow(np.angle(commands['focus']).reshape(p.samples_side,-1),origin='lower',
    extent=[-p.side_m/2,p.side_m/2,-p.side_m/2,p.side_m/2],cmap='twilight',vmin=-np.pi,vmax=np.pi)
ax[0].set(xlabel='Aperture x (m)',ylabel='Aperture y (m)',title=f'Spherical phase command at {p.frequency_GHz:g} GHz')
fig.colorbar(im,ax=ax[0],label='Phase (rad)')
for (key,curve),color in zip(profiles.items(),colors):
    ax[1].plot(xs,db(curve/gain0),label=key,color=color)
ax[1].axvline(p.rx_m[0],ls=':',color='black',label='target')
ax[1].set(xlabel=f'Receiver x (m), z = {p.rx_m[2]:g} m',ylabel='Gain / ideal target gain (dB)',
          ylim=(-45,1),title='Fixed commands, moving receiver')
ax[1].legend()
plt.show()
evidence['focus']={'x_m':xs.tolist(),'gain':{k:v.tolist() for k,v in profiles.items()},
    'phase':{k:np.angle(v).reshape(p.samples_side,-1)[::4,::4].tolist() for k,v in commands.items()},
    'target_gain':{k:float(abs(channel(v,p))**2) for k,v in commands.items()}}

## ۲. مساحت، نمونه‌برداری و تداخل همدوس

در مساحت ثابت، شبکهٔ میانی ریزتر به یک انتگرال همگرا می‌شود. پنل دوم مساحت فیزیکی را تغییر می‌دهد و دهانه را دوباره برای هدف تنظیم می‌کند؛ رابطهٔ خط‌چین میدان دور صرفاً مرجع مقایسه است. پنل آخر مسیر مستقیم فریس را با افت اضافی **توان** به‌اندازهٔ ۱۰ دسی‌بل اضافه و فاز مشترک فراسطح را نسبت به آن تغییر می‌دهد. مکان گره‌ها ثابت و مسیر فراسطح متمرکز می‌ماند. جمع میدان‌ها می‌تواند توان دریافتی را کم کند.

در این مقایسه، بهره‌های توان فرستنده و گیرنده برای هر دو مسیر مستقیم و مسیر فراسطح اعمال می‌شوند؛ پس دامنه شامل ریشهٔ حاصل‌ضرب و توان شامل خودِ حاصل‌ضرب بهره‌هاست. الگوی آنتنِ وابسته به جهت به تعریف بهره‌های جداگانه نیاز دارد.


In [ ]:
ns = [12,24,48,96,192]
grid_gain=[]
for n in ns:
    q=replace(p,samples_side=n); grid_gain.append(abs(channel(phase_command(q),q))**2)
sides=np.linspace(.12,.72,31); area_gain=[]
for side in sides:
    q=replace(p,side_m=float(side)); area_gain.append(abs(channel(phase_command(q),q))**2)
t=np.array(p.tx_m);r=np.array(p.rx_m)
rt=np.linalg.norm(t);rr=np.linalg.norm(r)
far_gain=p.tx_gain*p.rx_gain*p.efficiency*(t[2]/rt)*(r[2]/rr)*sides**4/(16*np.pi**2*rt**2*rr**2)
direct_distance=np.linalg.norm(t-r)
direct_amp=np.sqrt(p.tx_gain*p.rx_gain)*C0/(p.frequency_GHz*1e9)/(4*np.pi*direct_distance)*10**(-10/20)
phase=np.linspace(-180,180,361)
combined=abs(direct_amp+abs(h0)*np.exp(1j*np.deg2rad(phase)))**2
fig,ax=plt.subplots(1,3,figsize=(12.5,3.8),layout='constrained')
ax[0].plot(ns,np.array(grid_gain)/gain0,'o-',color=colors[0])
ax[0].set(xlabel='Samples per side',ylabel=f'Gain / {p.samples_side}-grid gain',title='Same area, finer quadrature')
ax[0].ticklabel_format(axis='y',style='plain',useOffset=False)
ax[1].plot(sides**2,db(area_gain),label='spherical sum',color=colors[0])
ax[1].plot(sides**2,db(far_gain),'--',label='far-field comparison',color=colors[1])
ax[1].set(xlabel='Physical area (m²)',ylabel='Channel power gain (dB)',title='Changing area');ax[1].legend()
ax[2].plot(phase,p.power_dBm+db(combined),color=colors[0])
ax[2].axhline(p.power_dBm+db(gain0),ls='--',color=colors[1],label='surface alone')
ax[2].set(xlabel='Relative common phase (deg)',ylabel='Received power (dBm)',title='Direct + assisted path');ax[2].legend()
plt.show()
evidence['power']={'samples':ns,'grid_gain':grid_gain,'side_m':sides.tolist(),'area_gain':area_gain,
    'far_gain':far_gain.tolist(),'phase_deg':phase.tolist(),'combined_gain':combined.tolist(),
    'direct_amplitude':float(direct_amp),'direct_excess_loss_dB':10.}

## ۳. تنظیم فاز، تأخیر زمانی واقعی نیست

فرمان ۲۸ گیگاهرتز را ثابت نگه دارید و فرکانس را با توان منبع یکسان تغییر دهید. مرجع تأخیر ایده‌آل اختلاف مسیر را در هر فرکانس خنثی می‌کند؛ دامنهٔ بی‌پاشندگی و تأخیر نسبی قابل تحقق را فرض می‌کند و پاسخ دستگاه شبیه‌سازی‌شده نیست. مکان قلهٔ جانبی با فرکانس جابه‌جا می‌شود. پنل فرکانس بهرهٔ کانال است، نه گذردهی پهن‌باند؛ نرخ پهن‌باند به چگالی طیفی توان و انتگرال نویز نیز نیاز دارد.

In [ ]:
frequencies=np.linspace(p.frequency_GHz-3,p.frequency_GHz+3,121)
fixed=np.array([abs(channel(commands['focus'],p,frequency_GHz=f))**2 for f in frequencies])
ideal=np.array([paths(p,frequency_GHz=f)[2].sum()**2 for f in frequencies])
wide_profiles={str(f):np.array([abs(channel(commands['focus'],p,(x,0,p.rx_m[2]),f))**2 for x in xs]) for f in (p.frequency_GHz-3,p.frequency_GHz,p.frequency_GHz+3)}
fig,ax=plt.subplots(1,2,figsize=(11.5,4),layout='constrained')
ax[0].plot(frequencies,db(fixed/gain0),color=colors[0],label=f'fixed phase at {p.frequency_GHz:g} GHz')
ax[0].plot(frequencies,db(ideal/gain0),'--',color=colors[1],label='ideal relative delay')
ax[0].set(xlabel='Frequency (GHz)',ylabel='Gain / ideal target gain (dB)',title='At the target');ax[0].legend()
for (key,curve),color in zip(wide_profiles.items(),colors):
    ax[1].plot(xs,db(curve/gain0),label=key+' GHz',color=color)
ax[1].set(xlabel=f'Receiver x (m), z = {p.rx_m[2]:g} m',ylabel='Gain / ideal target gain (dB)',
          ylim=(-35,1),title='Beam displacement');ax[1].legend()
plt.show()
evidence['bandwidth']={'GHz':frequencies.tolist(),'fixed_gain':fixed.tolist(),'delay_gain':ideal.tolist(),
    'profiles':{k:v.tolist() for k,v in wide_profiles.items()}}

## ۴. آمار خطا و هزینهٔ آموزش

پنل چپ میانگین دقیق توان را با ۲۵۶ تحقق بذرثابت خطای فاز گاوسی مستقل مقایسه می‌کند. میلهٔ خطا دو برابر خطای استاندارد میانگین نمونه است، نه صدک قطعی ارتباط. در پنل راست توان پایلوت بدون نویز برای کاربر در x = 8.37 m و z = 12 m بررسی می‌شود. مجموعهٔ K نقطه، بازهٔ x = 4–12 m را می‌پوشاند؛ برای K = 1 نقطهٔ x = 8 m به‌کار می‌رود. هر نامزد یک نماد از بلوک ۲۰۰ نمادی هزینه دارد. دیگر شرایط ثابت‌اند. این انتخاب جامع پرتو است، نه تخمین‌زن کانال.

In [ ]:
sigma=np.linspace(0,90,7);rng=np.random.default_rng(8);means=[];sem=[]
for s in sigma:
    trials=abs(np.exp(1j*rng.normal(0,np.deg2rad(s),(256,len(weights))))@weights)**2/gain0
    means.append(float(trials.mean()));sem.append(float(trials.std(ddof=1)/np.sqrt(len(trials))))
expected=error_expectation(weights,np.deg2rad(sigma))/gain0
train=training(p);counts=np.array([v['count'] for v in train])
fig,ax=plt.subplots(1,2,figsize=(11.5,4),layout='constrained')
ax[0].plot(sigma,expected,label='exact ensemble mean',color=colors[0])
ax[0].errorbar(sigma,means,yerr=2*np.array(sem),fmt='o',label='256 trials, ±2 SE',color=colors[1])
ax[0].set(xlabel='Phase-error standard deviation (deg)',ylabel='Mean gain / ideal gain',title='Independent phase errors');ax[0].legend()
ax[1].plot(counts,[v['gross_Mbps'] for v in train],'o--',label='gross AWGN benchmark',color=colors[1])
ax[1].plot(counts,[v['net_Mbps'] for v in train],'o-',label='after pilot time',color=colors[0])
ax[1].set(xlabel='Number of tested focus points K',ylabel='Rate (Mbps)',title='200-symbol coherence block');ax[1].legend()
plt.show()
evidence['training']={'rows':train,'coherence':200,'user_x_m':8.37,'sigma_deg':sigma.tolist(),
    'mean_ratio':expected.tolist(),'mc_ratio':means,'mc_SE':sem,'trials':256,'seed':8}
print('Best tested net-rate codebook:',max(train,key=lambda row:row['net_Mbps']))

## ۵. ذخیره، ارزیابی و نقد مدل

مرورگر از آرایه‌های ذخیره‌شده استفاده می‌کند و فقط رابطه‌های سادهٔ نویز و نرخ نمایش‌داده‌شده را دوباره محاسبه می‌کند. فرمان python scripts/validate_chapter08.py را جداگانه اجرا کنید: انتگرال تطبیقی پیوسته، پیاده‌سازی حلقه‌ای مستقل، نرمال‌سازی میدان دور، تقابل، کران همدوس، همانی‌های سینک و گاوسی، محاسبهٔ نرخ و ورودی نامعتبر بررسی می‌شوند.

یک توسعهٔ موجه را امتحان کنید: گیرندهٔ جابه‌جاشده، کتابخانهٔ حالت مختلط اندازه‌گیری‌شده، انتخاب پرتو نویزی یا چند کاربر. دادهٔ مبنا را نگه دارید و خطای مدل را از خطای انتگرال‌گیری جدا کنید. بررسی عددی این مدل، اعتبارسنجی سطح ساخته‌شده یا پیوند بی‌سیم اندازه‌گیری‌شده نیست.

In [ ]:
out=ROOT/'data/chapter-08';out.mkdir(parents=True,exist_ok=True)
(out/'reference.json').write_text(json.dumps(evidence,separators=(',',':'))+'\n',encoding='utf-8')
print('Saved reference.json; no full-wave simulation or measurement is represented.')